# Gas Classification

Classifies which of the 10 analytes is present from the S11 spectra.

SVC, GB, and GP classifiers with data augmentation (noise injection) applied during training.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# paths
FILE_PATH    = 'processed_data.pkl'
PRELIM_PATH  = 'preliminary_results.pkl'
RESULTS_PATH = 'classification_results.pkl'

# constants
RANDOM_STATE   = 42
KFOLD_N_SPLITS = 5

# GP memory management
GP_SUBSAMPLE = True   # set False to train GP on the full fold (needs >>8 GB RAM)
GP_MAX_TRAIN = 3000   # max training samples fed to GPC per fold when GP_SUBSAMPLE=True

# override preliminary settings (None -> use preliminary recommendation)
OVERRIDE_REPR       = 'full'
OVERRIDE_COMPONENTS = 10
OVERRIDE_NOISE      = 0.025

MODEL_NAMES = ['PCA + SVC', 'PCA + GB']#, 'PCA + GP']

plt.rcParams.update({
    'font.size'         : 10,
    'figure.figsize'    : (14, 5),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

## Load Dataset

In [ ]:
with open(FILE_PATH,   'rb') as f: df_all = pickle.load(f)
with open(PRELIM_PATH, 'rb') as f: prelim = pickle.load(f)

print(f'Total sweeps: {len(df_all):,}')
print(f'Gas types: {sorted(df_all["gas_type"].unique())}')
print(f'Per-gas counts:')
print(df_all['gas_type'].value_counts().sort_index().to_string())

In [ ]:
SENSOR_NAMES = [
    'Sensor A — GO/Nafion (S11)',
    'Sensor B — G/GO/PEDOT:PSS (S22)',
]

def _resolve(override, prelim_val):
    return override if override is not None else prelim_val

SENSOR_CONFIG = {
    SENSOR_NAMES[0]: {
        'feature_col' : 'features_a',
        'best_repr'   : _resolve(OVERRIDE_REPR,       prelim[SENSOR_NAMES[0]]['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim[SENSOR_NAMES[0]]['n_components']),
    },
    SENSOR_NAMES[1]: {
        'feature_col' : 'features_b',
        'best_repr'   : _resolve(OVERRIDE_REPR,       prelim[SENSOR_NAMES[1]]['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim[SENSOR_NAMES[1]]['n_components']),
    },
}
NOISE_LEVEL = _resolve(OVERRIDE_NOISE, prelim['noise_level'])

for sn, cfg in SENSOR_CONFIG.items():
    print(f'  {sn}: repr={cfg["best_repr"]}, n_comp={cfg["n_components"]}')
print(f'  NOISE_LEVEL = {NOISE_LEVEL}')

## Feature Representation

In [ ]:
N_FREQ_PTS = len(df_all['features_a'].iloc[0])
_i1 = int(N_FREQ_PTS * (6000 - 2000) / (8000 - 2000))   # 2-6 GHz upper index


def build_representation(X_raw: np.ndarray, repr_name: str) -> np.ndarray:
    if repr_name == 'full':
        return X_raw
    elif repr_name == 'band_limited':
        return X_raw[:, :_i1]
    elif repr_name == 'derivative':
        return np.gradient(X_raw, axis=1)
    else:
        raise ValueError(f"Unknown representation '{repr_name}'")

## Helper Functions

In [ ]:
def build_classifier(model_name, n_components):
    """Build a PCA + classifier pipeline."""
    if model_name == 'PCA + SVC':
        clf = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    elif model_name == 'PCA + GB':
        clf = GradientBoostingClassifier(random_state=RANDOM_STATE)
    elif model_name == 'PCA + GP':
        # GPC with RBF kernel; n_jobs=-1 for multi-class one-vs-rest parallelism
        clf = GaussianProcessClassifier(
            kernel=1.0 * RBF(1.0),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(model_name)
    return Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ('clf',    clf),
    ])


print('Helper functions defined.')

## Classification (K-Fold)

In [ ]:
all_clf_results = {}   # {sensor: {model: {metrics + raw predictions}}}

le = LabelEncoder()
le.fit(df_all['gas_type'])
n_classes = len(le.classes_)
print(f'Classes ({n_classes}): {le.classes_}')

for sensor_name, cfg in SENSOR_CONFIG.items():
    print(f'\n{"="*60}\n{sensor_name}\n{"="*60}')
    all_clf_results[sensor_name] = {}

    X_raw = np.stack(df_all[cfg['feature_col']].values)
    X     = build_representation(X_raw, cfg['best_repr'])
    y     = le.transform(df_all['gas_type'].values)
    nc    = cfg['n_components']

    skf   = StratifiedKFold(n_splits=KFOLD_N_SPLITS, shuffle=True,
                             random_state=RANDOM_STATE)
    model_preds = {
        m: {'y_true': [], 'y_pred': [], 'y_score': []}
        for m in MODEL_NAMES
    }

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
        print(f'  Fold {fold+1}/{KFOLD_N_SPLITS} ...')
        X_tr = X[tr_idx] + np.random.normal(0, NOISE_LEVEL, X[tr_idx].shape)
        X_te = X[te_idx]

        for model_name in MODEL_NAMES:
            print(f'    {model_name} ...', end=' ', flush=True)
            clf = build_classifier(model_name, nc)

            if model_name == 'PCA + GP' and GP_SUBSAMPLE and len(tr_idx) > GP_MAX_TRAIN:
                rng = np.random.default_rng(RANDOM_STATE + fold)
                # Stratified subsample to preserve class balance
                sub_idx = []
                for cls in np.unique(y[tr_idx]):
                    cls_mask = np.where(y[tr_idx] == cls)[0]
                    n_cls = max(1, round(GP_MAX_TRAIN * len(cls_mask) / len(tr_idx)))
                    chosen = rng.choice(cls_mask, min(n_cls, len(cls_mask)), replace=False)
                    sub_idx.append(chosen)
                sub_idx = np.concatenate(sub_idx)
                clf.fit(X_tr[sub_idx], y[tr_idx[sub_idx]])
            else:
                clf.fit(X_tr, y[tr_idx])

            y_pred  = clf.predict(X_te)
            y_score = clf.predict_proba(X_te)   # (n_test, n_classes)
            model_preds[model_name]['y_true'].extend(y[te_idx])
            model_preds[model_name]['y_pred'].extend(y_pred)
            model_preds[model_name]['y_score'].append(y_score)
            print('done')

    for model_name in MODEL_NAMES:
        y_true  = np.array(model_preds[model_name]['y_true'])
        y_pred  = np.array(model_preds[model_name]['y_pred'])
        y_score = np.vstack(model_preds[model_name]['y_score'])   # (N, n_classes)

        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        roc_auc    = roc_auc_score(y_true_bin, y_score,
                                   multi_class='ovr', average='macro')

        all_clf_results[sensor_name][model_name] = {
            'y_true'          : y_true,
            'y_pred'          : y_pred,
            'y_score'         : y_score,
            'confusion_matrix': confusion_matrix(y_true, y_pred),
            'accuracy'        : accuracy_score (y_true, y_pred),
            'f1'              : f1_score       (y_true, y_pred, average='macro'),
            'precision'       : precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall'          : recall_score   (y_true, y_pred, average='macro', zero_division=0),
            'roc_auc'         : roc_auc,
        }
        res = all_clf_results[sensor_name][model_name]
        print(f'  [{model_name}]  '
              f'Acc={res["accuracy"]:.4f}  F1={res["f1"]:.4f}  AUC={res["roc_auc"]:.4f}')

print('\nClassification complete.')

## Export Results

In [ ]:
# Summary DataFrame
clf_summary_rows = []
for sensor_name, model_results in all_clf_results.items():
    for model_name, res in model_results.items():
        clf_summary_rows.append({
            'Sensor'   : sensor_name.split('—')[0].strip(),
            'Model'    : model_name,
            'Accuracy' : round(res['accuracy'],  4),
            'Macro F1' : round(res['f1'],        4),
            'Precision': round(res['precision'], 4),
            'Recall'   : round(res['recall'],    4),
            'ROC-AUC'  : round(res['roc_auc'],   4),
        })

df_summary = pd.DataFrame(clf_summary_rows)
print(df_summary.to_string(index=False))

# Export
results_bundle = {
    'clf_results'  : all_clf_results,   # {sensor: {model: {metrics + raw preds}}}
    'label_encoder': le,
    'summary_df'   : df_summary,
    'sensor_config': SENSOR_CONFIG,
    'metadata': {
        'kfold_n_splits': KFOLD_N_SPLITS,
        'noise_level'   : NOISE_LEVEL,
        'source_file'   : FILE_PATH,
        'prelim_file'   : PRELIM_PATH,
        'model_names'   : MODEL_NAMES,
        'cv_scheme'     : '5-fold StratifiedKFold',
        'target'        : 'gas_type',
        'n_classes'     : n_classes,
        'class_names'   : list(le.classes_),
    },
}

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results_bundle, f)

print(f'\nSaved {RESULTS_PATH}')
print('Ready for 01_results.ipynb')

## Classification (LOCO)

In [ ]:
# Cell A: assign concentration rank (per gas)
# Rank 0 = baseline (0 uL); ranks 1-20 = non-zero levels sorted ascending.
# Ranks are per-gas so we compare relative dose position, not absolute volume.

df_all = df_all.copy()   # avoid SettingWithCopyWarning on shared DataFrame

conc_rank = np.zeros(len(df_all), dtype=int)

for gas in df_all['gas_type'].unique():
    gas_mask      = df_all['gas_type'] == gas
    nonzero_concs = sorted(
        c for c in df_all.loc[gas_mask, 'concentration_ul'].unique() if c > 0
    )
    for rank, conc_val in enumerate(nonzero_concs, start=1):
        rows = np.where(gas_mask & (df_all['concentration_ul'] == conc_val))[0]
        conc_rank[rows] = rank

df_all['conc_rank'] = conc_rank

print('conc_rank distribution (0 = baseline):')
cr_counts = pd.Series(conc_rank).value_counts().sort_index()
print(cr_counts.to_string())
print(f'\nTotal samples: {len(df_all):,}')
print(f'Baseline (rank 0): {(conc_rank == 0).sum():,}')
print(f'Non-zero (ranks 1–20): {(conc_rank > 0).sum():,}')

In [ ]:
# Cell B: assign fold IDs (sequential block assignment)
# Fold 0 -> ranks 1-4  (lowest doses - extrapolation)
# Fold 1 -> ranks 5-8
# Fold 2 -> ranks 9-12
# Fold 3 -> ranks 13-16
# Fold 4 -> ranks 17-20 (highest doses - extrapolation)
# Baseline (rank 0) always in training (fold_id = -1).

LOCO_N_FOLDS  = 5
LOCO_PER_FOLD = 4   # rank indices per fold

fold_id = np.full(len(df_all), -1, dtype=int)
for fold in range(LOCO_N_FOLDS):
    lo = fold * LOCO_PER_FOLD + 1
    hi = lo + LOCO_PER_FOLD
    fold_id[(conc_rank >= lo) & (conc_rank < hi)] = fold

df_all['loco_fold'] = fold_id

print('Samples per fold (test partition):')
for f in range(LOCO_N_FOLDS):
    n = (fold_id == f).sum()
    lo = f * LOCO_PER_FOLD + 1
    print(f'  Fold {f}  ranks {lo}–{lo + LOCO_PER_FOLD - 1}:  {n:,} test samples')
print(f'  Baseline (always train, fold=-1): {(fold_id == -1).sum():,} samples')

In [ ]:
# Cell C: run LOCO CV

all_loco_results = {}

for sensor_name, cfg in SENSOR_CONFIG.items():
    print(f'\n{"="*60}\n{sensor_name} — Concentration-Level Holdout CV\n{"="*60}')
    all_loco_results[sensor_name] = {}

    X_raw = np.stack(df_all[cfg['feature_col']].values)
    X     = build_representation(X_raw, cfg['best_repr'])
    y     = le.transform(df_all['gas_type'].values)
    nc    = cfg['n_components']

    model_preds = {
        m: {'y_true': [], 'y_pred': [], 'y_score': [], 'fold_metrics': []}
        for m in MODEL_NAMES
    }

    for fold in range(LOCO_N_FOLDS):
        te_idx = np.where(fold_id == fold)[0]
        tr_idx = np.where(fold_id != fold)[0]   # includes baseline (-1) + other folds

        lo = fold * LOCO_PER_FOLD + 1
        print(f'  Fold {fold+1}/{LOCO_N_FOLDS}  '
              f'(ranks {lo}–{lo + LOCO_PER_FOLD - 1})  '
              f'train={len(tr_idx):,}  test={len(te_idx):,}')

        X_tr = X[tr_idx] + np.random.normal(0, NOISE_LEVEL, X[tr_idx].shape)
        X_te = X[te_idx]
        y_tr = y[tr_idx]
        y_te = y[te_idx]

        for model_name in MODEL_NAMES:
            print(f'    {model_name} ...', end=' ', flush=True)
            clf = build_classifier(model_name, nc)
            clf.fit(X_tr, y_tr)

            y_pred  = clf.predict(X_te)
            y_score = clf.predict_proba(X_te)

            fold_acc = accuracy_score(y_te, y_pred)
            fold_f1  = f1_score(y_te, y_pred, average='macro')

            # Per-fold AUC: baseline class absent from test -> suppress the warning
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                fold_auc = roc_auc_score(
                    label_binarize(y_te, classes=np.arange(n_classes)),
                    y_score, multi_class='ovr', average='macro',
                )

            model_preds[model_name]['fold_metrics'].append({
                'fold'    : fold,
                'ranks'   : f'{lo}–{lo + LOCO_PER_FOLD - 1}',
                'accuracy': fold_acc,
                'f1'      : fold_f1,
                'roc_auc' : fold_auc,
                'n_test'  : len(te_idx),
            })
            model_preds[model_name]['y_true'].extend(y_te)
            model_preds[model_name]['y_pred'].extend(y_pred)
            model_preds[model_name]['y_score'].append(y_score)
            print(f'Acc={fold_acc:.4f}  F1={fold_f1:.4f}  AUC={fold_auc:.4f}')

    for model_name in MODEL_NAMES:
        y_true  = np.array(model_preds[model_name]['y_true'])
        y_pred  = np.array(model_preds[model_name]['y_pred'])
        y_score = np.vstack(model_preds[model_name]['y_score'])

        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        roc_auc    = roc_auc_score(y_true_bin, y_score,
                                   multi_class='ovr', average='macro')

        fm = pd.DataFrame(model_preds[model_name]['fold_metrics'])
        all_loco_results[sensor_name][model_name] = {
            'y_true'          : y_true,
            'y_pred'          : y_pred,
            'y_score'         : y_score,
            'confusion_matrix': confusion_matrix(y_true, y_pred),
            'accuracy'        : accuracy_score(y_true, y_pred),
            'f1'              : f1_score(y_true, y_pred, average='macro'),
            'precision'       : precision_score(y_true, y_pred,
                                    average='macro', zero_division=0),
            'recall'          : recall_score(y_true, y_pred,
                                    average='macro', zero_division=0),
            'roc_auc'         : roc_auc,
            'fold_metrics'    : model_preds[model_name]['fold_metrics'],
        }
        res = all_loco_results[sensor_name][model_name]
        print(f'\n  [{model_name}] Overall: '
              f'Acc={res["accuracy"]:.4f}  F1={res["f1"]:.4f}  AUC={res["roc_auc"]:.4f}')
        print(f'  Per-fold Acc: mean={fm["accuracy"].mean():.4f} ± {fm["accuracy"].std():.4f}')

print('\nLOCO classification complete.')

## Export Results

In [ ]:
# Cell D: export - reload pkl, append LOCO key, re-save

with open(RESULTS_PATH, 'rb') as f:
    results_bundle = pickle.load(f)

loco_summary = pd.DataFrame([
    {
        'Sensor'   : sn.split('—')[0].strip(),
        'Model'    : mn,
        'Accuracy' : round(r['accuracy'],  4),
        'Macro F1' : round(r['f1'],        4),
        'Precision': round(r['precision'], 4),
        'Recall'   : round(r['recall'],    4),
        'ROC-AUC'  : round(r['roc_auc'],   4),
    }
    for sn, mr in all_loco_results.items()
    for mn, r  in mr.items()
])

results_bundle['clf_results_loco'] = all_loco_results
results_bundle['loco_summary_df']  = loco_summary
results_bundle['metadata']['cv_scheme_loco'] = (
    f'{LOCO_N_FOLDS}-fold Concentration-Level Holdout '
    f'({LOCO_PER_FOLD} rank indices per fold)'
)

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results_bundle, f)

print(loco_summary.to_string(index=False))
print(f'\nAppended clf_results_loco → {RESULTS_PATH}')
print('Ready for 01_results.ipynb')